In [11]:
%%writefile surveillance_desk.py
import asyncio
import json
import random
import socket
import datetime
import time
from typing import Dict, Any, List
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import HTMLResponse
import yfinance as yf

app = FastAPI(title="Cross-Asset Surveillance & Electronic Risk Desk")

# Universe Definition
SYMBOLS = [
    "AAPL", "MSFT", "NVDA", "TSLA", "AMZN",
    "RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS",
    "EURUSD=X", "GBPUSD=X", "USDJPY=X", "USDINR=X",
    "GC=F", "SI=F", "CL=F", "NG=F"
]

ASSET_CLASS_MAP = {
    "AAPL": "US_Equities", "MSFT": "US_Equities", "NVDA": "US_Equities", "TSLA": "US_Equities", "AMZN": "US_Equities",
    "RELIANCE.NS": "Indian_NSE", "TCS.NS": "Indian_NSE", "INFY.NS": "Indian_NSE", "HDFCBANK.NS": "Indian_NSE",
    "EURUSD=X": "Forex", "GBPUSD=X": "Forex", "USDJPY=X": "Forex", "USDINR=X": "Forex",
    "GC=F": "Commodities", "SI=F": "Commodities", "CL=F": "Commodities", "NG=F": "Commodities"
}

# Runtime States
DESK_MODE = "SIMULATION" # Options: "LIVE" or "SIMULATION"
PRICES: Dict[str, float] = {}
BASELINES: Dict[str, Dict[str, float]] = {}
VOLUMES: Dict[str, int] = {}
CONTROLS: Dict[str, Dict[str, Any]] = {}
ORDER_BOOKS: Dict[str, Dict[str, List[Dict[str, Any]]]] = {}
MSG_SEQ_NUM = 1000

# Initialize default baselines
for sym in SYMBOLS:
    BASELINES[sym] = {"atr_14": 2.50, "mean_vol_1m": 25000, "std_vol_1m": 6000}
    PRICES[sym] = 100.0
    VOLUMES[sym] = 25000
    CONTROLS[sym] = {
        "status": "NORMAL",
        "clip_cap": 2500,
        "throttled": False,
        "throttle_cooldown": 0
    }
    ORDER_BOOKS[sym] = {"bids": [], "asks": []}

def generate_l2_depth(sym: str, mid_price: float) -> Dict[str, List[Dict[str, Any]]]:
    tick_size = 0.0001 if "=X" in sym else (0.25 if mid_price > 1000 else 0.05)
    bids, asks = [], []
    for i in range(1, 6):
        b_price = round(mid_price - (i * tick_size), 4 if "=X" in sym else 2)
        bids.append({"price": b_price, "qty": random.randint(250, 4200), "orders": random.randint(2, 14)})
        a_price = round(mid_price + (i * tick_size), 4 if "=X" in sym else 2)
        asks.append({"price": a_price, "qty": random.randint(250, 4200), "orders": random.randint(2, 14)})
    return {"bids": bids, "asks": asks}

def format_fix_message(sym: str, price: float) -> str:
    global MSG_SEQ_NUM
    MSG_SEQ_NUM += 1
    t_stamp = datetime.datetime.utcnow().strftime("%H:%M:%S.%f")[:-3]
    m_type = random.choice(["D", "D", "F", "8"])
    side = random.choice(["1", "2"])
    qty = random.choice([100, 250, 500, 1000, 2500])
    p_str = f"{price:.4f}" if "=X" in sym else f"{price:.2f}"
    return f"8=FIX.4.2|35={m_type}|34={MSG_SEQ_NUM}|52={t_stamp}|55={sym}|54={side}|38={qty}|44={p_str}"

def fetch_latest_market_prices():
    """Fetch exact latest closing / live prices using yfinance"""
    print("Fetching latest closing prices from exchange...")
    try:
        tickers = yf.Tickers(" ".join(SYMBOLS))
        for sym in SYMBOLS:
            ticker = tickers.tickers[sym]
            hist = ticker.history(period="5d")
            if not hist.empty:
                last_close = round(float(hist["Close"].iloc[-1]), 4)
                PRICES[sym] = last_close
                high_low = (hist["High"] - hist["Low"]).mean()
                BASELINES[sym]["atr_14"] = round(float(high_low), 4) if high_low > 0 else 1.0
                mean_vol = hist["Volume"].mean() / 375
                BASELINES[sym]["mean_vol_1m"] = int(mean_vol) if mean_vol > 0 else 10000
                BASELINES[sym]["std_vol_1m"] = int(BASELINES[sym]["mean_vol_1m"] * 0.25)
                VOLUMES[sym] = BASELINES[sym]["mean_vol_1m"]
            ORDER_BOOKS[sym] = generate_l2_depth(sym, PRICES[sym])
    except Exception as e:
        print(f"Market fetch fallback: {e}")
        for sym in SYMBOLS:
            ORDER_BOOKS[sym] = generate_l2_depth(sym, PRICES[sym])

fetch_latest_market_prices()

class ConnectionManager:
    def __init__(self):
        self.active_connections: list[WebSocket] = []

    async def connect(self, websocket: WebSocket):
        await websocket.accept()
        self.active_connections.append(websocket)

    def disconnect(self, websocket: WebSocket):
        if websocket in self.active_connections:
            self.active_connections.remove(websocket)

    async def broadcast(self, message: dict):
        for connection in self.active_connections:
            try:
                await connection.send_text(json.dumps(message))
            except Exception:
                pass

manager = ConnectionManager()

def evaluate_tick_anomalies(sym: str, price: float, vol_1m: int, ctr: float, life_ms: int):
    base = BASELINES[sym]
    atr = base["atr_14"]
    mean_v = base["mean_vol_1m"]
    std_v = base["std_vol_1m"]
    
    z_score = round((vol_1m - mean_v) / (std_v if std_v > 0 else 1), 2)
    price_diff = abs(price - base.get("ref_price", price))
    
    if price_diff > (atr * 2.5):
        CONTROLS[sym]["status"] = "KILL_SWITCH"
        return z_score, {
            "type": "ALERT", "severity": "CRITICAL", "symbol": sym,
            "typology": "Flash Price Dislocation",
            "msg": f"Price drifted {price_diff:.2f} > 2.5x ATR ({atr:.2f})",
            "control": "⛔ KILL-SWITCH: Session locked & order book purged."
        }
    
    if ctr >= 90.0 and life_ms < 200:
        CONTROLS[sym]["throttled"] = True
        CONTROLS[sym]["throttle_cooldown"] = 10
        return z_score, {
            "type": "ALERT", "severity": "HIGH", "symbol": sym,
            "typology": "Spoofing / Layering Signature",
            "msg": f"CTR {ctr}% with avg life {life_ms}ms",
            "control": "⏳ GATEWAY THROTTLED: 5 msgs/sec enforced."
        }
        
    if z_score >= 4.0:
        CONTROLS[sym]["clip_cap"] = 500
        return z_score, {
            "type": "ALERT", "severity": "MEDIUM", "symbol": sym,
            "typology": "Abnormal Volume Burst",
            "msg": f"1m Volume {vol_1m:,} at Z-Score +{z_score}σ",
            "control": "📉 CLIP-CAP CLAMPED: Max allowable lot reduced to 500."
        }
        
    return z_score, None

async def market_telemetry_loop():
    global DESK_MODE
    while True:
        if DESK_MODE == "SIMULATION":
            sym = random.choice(SYMBOLS)
            meta = BASELINES[sym]
            
            step = random.gauss(0, meta["atr_14"] * 0.015)
            PRICES[sym] = round(max(PRICES[sym] + step, 0.01), 4 if "X" in sym else 2)
            
            vol = int(max(random.gauss(meta["mean_vol_1m"], meta["std_vol_1m"]), 100))
            VOLUMES[sym] = vol
            ctr = round(random.uniform(40.0, 75.0), 1)
            life = random.randint(300, 1500)
            
            ORDER_BOOKS[sym] = generate_l2_depth(sym, PRICES[sym])
            fix_line = format_fix_message(sym, PRICES[sym])
            
            z_score, alert = evaluate_tick_anomalies(sym, PRICES[sym], vol, ctr, life)
            
            if CONTROLS[sym]["throttle_cooldown"] > 0:
                CONTROLS[sym]["throttle_cooldown"] -= 1
                if CONTROLS[sym]["throttle_cooldown"] == 0:
                    CONTROLS[sym]["throttled"] = False
                    
            tick = {
                "type": "TICK",
                "symbol": sym,
                "asset_class": ASSET_CLASS_MAP[sym],
                "price": PRICES[sym],
                "volume": VOLUMES[sym],
                "z_score": z_score,
                "control_state": CONTROLS[sym]["status"],
                "clip_cap": CONTROLS[sym]["clip_cap"],
                "throttled": CONTROLS[sym]["throttled"],
                "book": ORDER_BOOKS[sym],
                "fix_msg": fix_line,
                "mode": "SIMULATION"
            }
            await manager.broadcast(tick)
            if alert:
                await manager.broadcast(alert)
        else:
            await manager.broadcast({
                "type": "LIVE_HEARTBEAT",
                "mode": "LIVE",
                "market_status": "CLOSED (WEEKEND FROZEN)",
                "prices": {s: PRICES[s] for s in SYMBOLS}
            })
            
        await asyncio.sleep(0.35)

@app.on_event("startup")
async def startup_event():
    for sym in SYMBOLS:
        BASELINES[sym]["ref_price"] = PRICES[sym]
    asyncio.create_task(market_telemetry_loop())

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    global DESK_MODE
    await manager.connect(websocket)
    snapshot = {
        "type": "SNAPSHOT",
        "mode": DESK_MODE,
        "instruments": [
            {
                "symbol": s,
                "asset_class": ASSET_CLASS_MAP[s],
                "price": PRICES[s],
                "volume": VOLUMES[s],
                "z_score": 0.0,
                "control_state": CONTROLS[s]["status"],
                "clip_cap": CONTROLS[s]["clip_cap"],
                "throttled": CONTROLS[s]["throttled"],
                "book": ORDER_BOOKS[s]
            } for s in SYMBOLS
        ]
    }
    await websocket.send_text(json.dumps(snapshot))
    try:
        while True:
            data = await websocket.receive_text()
            cmd = json.loads(data)
            action = cmd.get("action")
            
            if action == "TOGGLE_MODE":
                DESK_MODE = cmd.get("mode", "SIMULATION")
                await manager.broadcast({"type": "MODE_CHANGE", "mode": DESK_MODE})
                
            elif action == "RESET_KILL_SWITCH":
                sym = cmd.get("symbol")
                if sym in CONTROLS:
                    CONTROLS[sym]["status"] = "NORMAL"
                    BASELINES[sym]["ref_price"] = PRICES[sym]
                    await manager.broadcast({"type": "CONTROL_UPDATE", "symbol": sym, "status": "NORMAL"})
                    
            elif action == "UPDATE_CLIP_CAP":
                sym = cmd.get("symbol")
                val = cmd.get("value")
                if sym in CONTROLS:
                    CONTROLS[sym]["clip_cap"] = int(val)
                    await manager.broadcast({"type": "CONTROL_UPDATE", "symbol": sym, "clip_cap": int(val)})
                    
            elif action == "SIMULATE_ANOMALY":
                sym = cmd.get("symbol")
                kind = cmd.get("kind")
                if sym in SYMBOLS:
                    if kind == "FLASH_CRASH":
                        PRICES[sym] -= (BASELINES[sym]["atr_14"] * 3.2)
                    elif kind == "VOLUME_BURST":
                        VOLUMES[sym] = BASELINES[sym]["mean_vol_1m"] + (BASELINES[sym]["std_vol_1m"] * 5)
                    elif kind == "SPOOFING":
                        pass
                    z, alert = evaluate_tick_anomalies(
                        sym, PRICES[sym], VOLUMES[sym], 
                        96.0 if kind == "SPOOFING" else 50.0,
                        80 if kind == "SPOOFING" else 600
                    )
                    if alert:
                        await manager.broadcast(alert)
    except WebSocketDisconnect:
        manager.disconnect(websocket)

@app.get("/")
def get_dashboard():
    return HTMLResponse("""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=no">
<title>Cross-Asset Surveillance & Electronic Risk Desk</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<style>
  :root { --bg: #090d16; --panel: #111827; --border: #1f2937; --text: #e5e7eb; --accent: #38bdf8; --red: #ef4444; --green: #10b981; --amber: #f59e0b; }
  body { margin:0; background: var(--bg); color: var(--text); font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, monospace; overflow-x: hidden; }
  .header { display: flex; justify-content: space-between; align-items: center; padding: 12px 24px; background: #0f172a; border-bottom: 1px solid var(--border); flex-wrap: wrap; gap: 8px; }
  .title { font-size: 17px; font-weight: 700; color: #fff; letter-spacing: 0.5px; }
  .mode-switch { display: flex; align-items: center; gap: 6px; background: #000; padding: 3px; border-radius: 6px; border: 1px solid var(--border); }
  .mode-btn { border: none; padding: 5px 10px; font-size: 11px; font-weight: 700; border-radius: 4px; cursor: pointer; color: #888; background: transparent; }
  .mode-btn.active-sim { background: var(--accent); color: #000; }
  .mode-btn.active-live { background: var(--green); color: #000; }
  .strobe-bar { display: none; padding: 8px; background: var(--red); color: white; text-align: center; font-weight: bold; font-size: 13px; letter-spacing: 1px; animation: strobe 0.8s infinite; }
  @keyframes strobe { 0% { opacity: 1; } 50% { opacity: 0.4; } 100% { opacity: 1; } }
  
  .grid { display: grid; grid-template-columns: 3fr 1.8fr; gap: 16px; padding: 16px; }
  @media (max-width: 900px) {
    .grid { grid-template-columns: 1fr; }
    .table-container { overflow-x: auto; }
  }

  .panel { background: var(--panel); border: 1px solid var(--border); border-radius: 8px; padding: 14px; }
  table { width: 100%; border-collapse: collapse; font-size: 13px; }
  th { text-align: left; padding: 8px; color: #9ca3af; border-bottom: 1px solid var(--border); font-size: 11px; text-transform: uppercase; }
  td { padding: 8px; border-bottom: 1px solid #1a2234; cursor: pointer; }
  tr:hover td { background: #1e293b; }
  tr.selected td { background: #1e293b; border-left: 3px solid var(--accent); }
  .badge { padding: 2px 6px; border-radius: 4px; font-size: 10px; font-weight: 700; }
  .b-normal { background: rgba(16,185,129,0.2); color: var(--green); }
  .b-kill { background: rgba(239,68,68,0.2); color: var(--red); }
  .b-throttle { background: rgba(245,158,11,0.2); color: var(--amber); }
  .alert-feed { height: 180px; overflow-y: auto; display: flex; flex-direction: column; gap: 8px; }
  .alert-card { padding: 10px; border-radius: 6px; font-size: 12px; border-left: 4px solid var(--accent); background: #172033; }
  .alert-card.CRITICAL { border-color: var(--red); }
  .alert-card.HIGH { border-color: var(--amber); }
  .btn { background: #1e293b; border: 1px solid var(--border); color: #fff; padding: 5px 10px; border-radius: 4px; cursor: pointer; font-size: 11px; font-weight: 600; }
  .btn:hover { background: #334155; }
  .btn-danger { background: #7f1d1d; border-color: var(--red); color: #fca5a5; }
  .btn-danger:hover { background: #991b1b; }
  .market-pill { font-size: 11px; padding: 3px 8px; border-radius: 4px; font-weight: 600; }
  .mp-closed { background: #78350f; color: #fde68a; border: 1px solid var(--amber); }
  .mp-open { background: #064e3b; color: #a7f3d0; border: 1px solid var(--green); }

  /* L2 Order Book Ladder */
  .book-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; font-size: 11px; font-family: monospace; margin-top: 6px; }
  .book-row { display: flex; justify-content: space-between; position: relative; padding: 3px 6px; border-radius: 2px; margin-bottom: 2px; cursor: pointer; }
  .book-row:hover { background: #26334a; }
  .depth-bar { position: absolute; top: 0; bottom: 0; opacity: 0.22; z-index: 1; border-radius: 2px; }
  .bid-bar { right: 0; background: var(--green); }
  .ask-bar { left: 0; background: var(--red); }
  .book-txt { position: relative; z-index: 2; }
  .bid-c { color: #34d399; font-weight: bold; }
  .ask-c { color: #f87171; font-weight: bold; }
  
  /* Microstructure Metrics Bar */
  .micro-metrics { display: flex; justify-content: space-between; padding: 5px 8px; background: #0c1220; border-radius: 4px; font-size: 11px; margin-bottom: 6px; }

  /* FIX Protocol Stream Terminal */
  .fix-terminal { height: 110px; overflow-y: scroll; background: #000; border: 1px solid var(--border); border-radius: 6px; padding: 6px 8px; font-family: 'Courier New', monospace; font-size: 10px; color: #4ade80; }
  .fix-row { margin-bottom: 3px; word-break: break-all; }
  .tag-d { color: #38bdf8; font-weight: bold; }
  .tag-f { color: #f87171; font-weight: bold; }
  .tag-8 { color: #facc15; font-weight: bold; }
</style>
</head>
<body>

<div id="strobe" class="strobe-bar">🚨 RISK BREACH DETECTED: AUDIT ACTIONS ENFORCED</div>

<div class="header">
  <div>
    <span class="title">CROSS-ASSET MARKET SURVEILLANCE & RISK DESK</span>
    <span id="market-status-pill" class="market-pill mp-closed" style="margin-left: 10px;">MARKET CLOSED (SUNDAY)</span>
  </div>
  <div style="display:flex; align-items:center; gap: 8px;">
    <!-- Feature: Export Audit Trail Button -->
    <button class="btn" style="border-color:#10b981; color:#a7f3d0;" onclick="exportAuditTrail()">📥 EXPORT AUDIT</button>
    <!-- Feature: Mute / Unmute Button -->
    <button id="btn-audio" class="btn" style="border-color: #ef4444; color: #fca5a5;" onclick="toggleAudio()">🔇 MUTED</button>
    <div class="mode-switch">
      <button id="btn-mode-live" class="mode-btn" onclick="setDeskMode('LIVE')">🟢 LIVE FEED</button>
      <button id="btn-mode-sim" class="mode-btn active-sim" onclick="setDeskMode('SIMULATION')">⚡ SIMULATOR</button>
    </div>
    <span id="clock" style="font-size: 11px; color: #94a3b8; font-weight: 600;">--:--:--</span>
  </div>
</div>

<div class="grid">
  <!-- Left Side: Perimeter Instruments Table -->
  <div class="panel table-container">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:12px;">
      <span style="font-weight:700; font-size:14px;">PERIMETER INSTRUMENTS (17 GLOBAL)</span>
      <span style="font-size:11px; color:#64748b;">Sub-second Live Ticks</span>
    </div>
    <table>
      <thead>
        <tr><th>Symbol</th><th>Class</th><th>Last Price</th><th>1M Vol</th><th>Z-Score</th><th>Control State</th><th>Action</th></tr>
      </thead>
      <tbody id="instrument-rows"></tbody>
    </table>
  </div>

  <!-- Right Side: L2 Depth + Telemetry + Controls + FIX Feed + Event Log -->
  <div style="display:flex; flex-direction:column; gap:14px;">
    
    <!-- Feature: L2 Order Book Ladder with Real-Time OBI Analytics -->
    <div class="panel">
      <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
        <span style="font-weight:700; font-size:13px;">LEVEL-2 ORDER BOOK DEPTH (TOP 5 LADDER)</span>
        <span id="book-sym" style="font-size:11px; color:var(--accent); font-weight:700;">AAPL</span>
      </div>
      
      <!-- Real-Time Microstructure Bar -->
      <div class="micro-metrics">
        <div>Spread: <b id="lbl-spread">0.05</b> (<span id="lbl-bps">1.8 bps</span>)</div>
        <div>Order Imbalance (OBI): <b id="lbl-obi" style="color:var(--green);">+14.2%</b></div>
        <div>Depth: <b id="lbl-depth">21,400</b> shs</div>
      </div>

      <div class="book-grid">
        <div>
          <div style="color:#64748b; font-size:10px; display:flex; justify-content:space-between; margin-bottom:4px;">
            <span>ORDERS | QTY</span><span>BID</span>
          </div>
          <div id="bids-container"></div>
        </div>
        <div>
          <div style="color:#64748b; font-size:10px; display:flex; justify-content:space-between; margin-bottom:4px;">
            <span>ASK</span><span>QTY | ORDERS</span>
          </div>
          <div id="asks-container"></div>
        </div>
      </div>
    </div>

    <!-- Telemetry Chart Panel -->
    <div class="panel">
      <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:8px;">
        <span style="font-weight:700; font-size:13px;" id="chart-title">TELEMETRY: AAPL</span>
        <span id="chart-sub" style="font-size:11px; color:var(--accent);">Sub-second Ticks</span>
      </div>
      <div style="height:150px;"><canvas id="telemetryChart"></canvas></div>
    </div>

    <!-- Active Risk Controls -->
    <div class="panel">
      <span style="font-weight:700; font-size:13px; display:block; margin-bottom:8px;">ACTIVE RISK MITIGATION CONTROLS</span>
      <div style="font-size:12px; display:grid; grid-template-columns: 1fr 1fr; gap:8px;">
        <div>Focus: <b id="ctrl-symbol">AAPL</b></div>
        <div>Session: <span id="ctrl-state" class="badge b-normal">NORMAL</span></div>
        <div>Gateway: <span id="ctrl-throttle" class="badge b-normal">NORMAL (UNLIMITED)</span></div>
        <div>Clip-Cap: <b id="ctrl-cap">2,500</b> units</div>
      </div>
      <div style="margin-top:12px; display:flex; gap:8px; flex-wrap:wrap;">
        <button class="btn btn-danger" onclick="resetKillSwitch()">Reset Kill-Switch</button>
        <button class="btn" onclick="adjustClipCap(500)">Clamp Clip 500</button>
        <button class="btn" onclick="adjustClipCap(2500)">Restore 2,500</button>
      </div>
      <div style="margin-top:12px; border-top: 1px solid var(--border); padding-top: 10px;">
        <span style="font-size:11px; color:#9ca3af; display:block; margin-bottom:6px;">INJECT ANOMALIES (STRESS TEST)</span>
        <div style="display:flex; gap:6px; flex-wrap:wrap;">
          <button class="btn" style="border-color:var(--red);" onclick="injectAnomaly('FLASH_CRASH')">Flash Crash</button>
          <button class="btn" style="border-color:var(--amber);" onclick="injectAnomaly('VOLUME_BURST')">Vol Burst</button>
          <button class="btn" style="border-color:var(--accent);" onclick="injectAnomaly('SPOOFING')">Spoof Loop</button>
        </div>
      </div>
    </div>

    <!-- Feature: Live FIX 4.2 Raw Protocol Stream Terminal -->
    <div class="panel">
      <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
        <span style="font-weight:700; font-size:13px;">RAW FIX 4.2 PROTOCOL STREAM (GATEWAY)</span>
        <span style="font-size:10px; color:#4ade80;">Tag 35=D/F/8</span>
      </div>
      <div class="fix-terminal" id="fix-feed"></div>
    </div>

    <!-- Event Dispatch Log -->
    <div class="panel">
      <span style="font-weight:700; font-size:13px; display:block; margin-bottom:8px;">SURVEILLANCE EVENT DISPATCH LOG</span>
      <div class="alert-feed" id="alert-feed"></div>
    </div>
  </div>
</div>

<script>
let ws;
let currentSymbol = "AAPL";
let deskMode = "SIMULATION";
let audioEnabled = false;
let booksCache = {};
let auditHistory = [];

const chartCtx = document.getElementById('telemetryChart').getContext('2d');
const chartData = { labels: [], datasets: [{ label: 'Price', data: [], borderColor: '#38bdf8', borderWidth: 2, tension: 0.2, fill: false }] };
const chart = new Chart(chartCtx, {
  type: 'line',
  data: chartData,
  options: {
    responsive: true,
    maintainAspectRatio: false,
    plugins: { legend: { display: false } },
    scales: { x: { display: false }, y: { grid: { color: '#1e293b' }, ticks: { color: '#94a3b8' } } }
  }
});

function toggleAudio() {
  audioEnabled = !audioEnabled;
  const btn = document.getElementById('btn-audio');
  btn.innerText = audioEnabled ? "🔊 AUDIO ON" : "🔇 MUTED";
  btn.style.borderColor = audioEnabled ? "#38bdf8" : "#ef4444";
  btn.style.color = audioEnabled ? "#38bdf8" : "#fca5a5";
  if (!audioEnabled && 'speechSynthesis' in window) {
    window.speechSynthesis.cancel();
  }
}

function speakAlert(text) {
  if (!audioEnabled) return;
  if ('speechSynthesis' in window) {
    window.speechSynthesis.cancel();
    const utter = new SpeechSynthesisUtterance(text);
    utter.rate = 1.05;
    window.speechSynthesis.speak(utter);
  }
}

function updateClock() {
  document.getElementById('clock').innerText = new Date().toLocaleTimeString();
}
setInterval(updateClock, 1000);
updateClock();

function renderOrderBook(book) {
  const bBox = document.getElementById('bids-container');
  const aBox = document.getElementById('asks-container');
  if (!bBox || !aBox || !book || !book.bids || !book.asks) return;

  let totalBid = book.bids.reduce((a, b) => a + b.qty, 0);
  let totalAsk = book.asks.reduce((a, b) => a + b.qty, 0);
  let maxQ = Math.max(...book.bids.map(b => b.qty), ...book.asks.map(a => a.qty), 3000);

  // Microstructure Bar Calculations
  let obi = ((totalBid - totalAsk) / (totalBid + totalAsk)) * 100;
  const obiEl = document.getElementById('lbl-obi');
  if (obiEl) {
    obiEl.innerText = `${obi >= 0 ? '+' : ''}${obi.toFixed(1)}%`;
    obiEl.style.color = obi >= 0 ? 'var(--green)' : 'var(--red)';
  }

  let spread = Math.abs(book.asks[0].price - book.bids[0].price);
  let bps = ((spread / book.bids[0].price) * 10000).toFixed(1);
  const spreadEl = document.getElementById('lbl-spread');
  const bpsEl = document.getElementById('lbl-bps');
  const depthEl = document.getElementById('lbl-depth');
  if (spreadEl) spreadEl.innerText = spread.toFixed(currentSymbol.includes('=X') ? 4 : 2);
  if (bpsEl) bpsEl.innerText = `${bps} bps`;
  if (depthEl) depthEl.innerText = (totalBid + totalAsk).toLocaleString();

  bBox.innerHTML = book.bids.map(b => `
    <div class="book-row">
      <div class="depth-bar bid-bar" style="width: ${(b.qty/maxQ)*100}%;"></div>
      <span class="book-txt">${b.orders}o | ${b.qty.toLocaleString()}</span>
      <span class="book-txt bid-c">${b.price.toFixed(currentSymbol.includes('=X')?4:2)}</span>
    </div>
  `).join('');

  aBox.innerHTML = book.asks.map(a => `
    <div class="book-row">
      <div class="depth-bar ask-bar" style="width: ${(a.qty/maxQ)*100}%;"></div>
      <span class="book-txt ask-c">${a.price.toFixed(currentSymbol.includes('=X')?4:2)}</span>
      <span class="book-txt">${a.qty.toLocaleString()} | ${a.orders}o</span>
    </div>
  `).join('');
}

function appendFixMessage(msg) {
  const feed = document.getElementById('fix-feed');
  if (!feed) return;
  const line = document.createElement('div');
  line.className = 'fix-row';
  
  let formatted = msg;
  if (msg.includes('35=D')) formatted = msg.replace('35=D', '<span class="tag-d">35=D (NewOrder)</span>');
  else if (msg.includes('35=F')) formatted = msg.replace('35=F', '<span class="tag-f">35=F (Cancel)</span>');
  else if (msg.includes('35=8')) formatted = msg.replace('35=8', '<span class="tag-8">35=8 (Execution)</span>');
  
  line.innerHTML = formatted;
  feed.insertBefore(line, feed.firstChild);
  if (feed.childNodes.length > 25) feed.removeChild(feed.lastChild);
}

function connectWS() {
  const protocol = window.location.protocol === 'https:' ? 'wss:' : 'ws:';
  ws = new WebSocket(`${protocol}//${window.location.host}/ws`);
  
  ws.onmessage = (e) => {
    const data = JSON.parse(e.data);
    if (data.type === "SNAPSHOT") {
      setModeUI(data.mode);
      renderTable(data.instruments);
      data.instruments.forEach(inst => {
        if (inst.book) booksCache[inst.symbol] = inst.book;
      });
      if (booksCache[currentSymbol]) renderOrderBook(booksCache[currentSymbol]);
    } else if (data.type === "TICK") {
      updateRow(data);
      if (data.book) {
        booksCache[data.symbol] = data.book;
      }
      if (data.fix_msg) {
        appendFixMessage(data.fix_msg);
      }
      if (data.symbol === currentSymbol) {
        if (data.book) renderOrderBook(data.book);
        chartData.labels.push("");
        chartData.datasets[0].data.push(data.price);
        if (chartData.labels.length > 25) { chartData.labels.shift(); chartData.datasets[0].data.shift(); }
        chart.update('none');
        document.getElementById('ctrl-state').innerText = data.control_state;
        document.getElementById('ctrl-state').className = `badge ${data.control_state === 'NORMAL' ? 'b-normal' : 'b-kill'}`;
        document.getElementById('ctrl-throttle').innerText = data.throttled ? "THROTTLED (5 msg/s)" : "NORMAL (UNLIMITED)";
        document.getElementById('ctrl-throttle').className = `badge ${data.throttled ? 'b-throttle' : 'b-normal'}`;
        document.getElementById('ctrl-cap').innerText = data.clip_cap.toLocaleString();
      }
    } else if (data.type === "ALERT") {
      auditHistory.push({ time: new Date().toISOString(), ...data });
      addAlertCard(data);
      triggerStrobe();
      speakAlert(`${data.typology} on ${data.symbol}`);
    } else if (data.type === "MODE_CHANGE") {
      setModeUI(data.mode);
    }
  };
  ws.onclose = () => setTimeout(connectWS, 1500);
}

function renderTable(instruments) {
  const tbody = document.getElementById('instrument-rows');
  tbody.innerHTML = "";
  instruments.forEach(inst => {
    const tr = document.createElement('tr');
    tr.id = `row-${inst.symbol}`;
    if (inst.symbol === currentSymbol) tr.className = "selected";
    tr.onclick = () => selectSymbol(inst.symbol);
    tr.innerHTML = `
      <td><b>${inst.symbol}</b></td>
      <td style="color:#94a3b8; font-size:11px;">${inst.asset_class}</td>
      <td id="p-${inst.symbol}"><b>${inst.price.toFixed(inst.symbol.includes('=X') ? 4 : 2)}</b></td>
      <td id="v-${inst.symbol}">${inst.volume.toLocaleString()}</td>
      <td id="z-${inst.symbol}">${inst.z_score >= 0 ? '+' : ''}${inst.z_score}σ</td>
      <td><span id="cs-${inst.symbol}" class="badge ${inst.control_state === 'NORMAL' ? 'b-normal' : 'b-kill'}">${inst.control_state}</span></td>
      <td><button class="btn" style="padding:2px 8px;" onclick="event.stopPropagation(); selectSymbol('${inst.symbol}')">Inspect</button></td>
    `;
    tbody.appendChild(tr);
  });
}

function updateRow(tick) {
  const pElem = document.getElementById(`p-${tick.symbol}`);
  const vElem = document.getElementById(`v-${tick.symbol}`);
  const zElem = document.getElementById(`z-${tick.symbol}`);
  const csElem = document.getElementById(`cs-${tick.symbol}`);
  if (pElem) pElem.innerText = tick.price.toFixed(tick.symbol.includes('=X') ? 4 : 2);
  if (vElem) vElem.innerText = tick.volume.toLocaleString();
  if (zElem) zElem.innerText = `${tick.z_score >= 0 ? '+' : ''}${tick.z_score}σ`;
  if (csElem) {
    csElem.innerText = tick.control_state;
    csElem.className = `badge ${tick.control_state === 'NORMAL' ? 'b-normal' : 'b-kill'}`;
  }
}

function selectSymbol(sym) {
  currentSymbol = sym;
  document.querySelectorAll('#instrument-rows tr').forEach(r => r.classList.remove('selected'));
  const row = document.getElementById(`row-${sym}`);
  if (row) row.classList.add('selected');
  document.getElementById('chart-title').innerText = `TELEMETRY: ${sym}`;
  document.getElementById('ctrl-symbol').innerText = sym;
  document.getElementById('book-sym').innerText = sym;
  if (booksCache[sym]) renderOrderBook(booksCache[sym]);
  chartData.labels = [];
  chartData.datasets[0].data = [];
  chart.update();
}

function addAlertCard(alert) {
  const feed = document.getElementById('alert-feed');
  const card = document.createElement('div');
  card.className = `alert-card ${alert.severity}`;
  card.innerHTML = `
    <div style="display:flex; justify-content:space-between; font-weight:700;">
      <span>[${alert.severity}] ${alert.typology}</span>
      <span style="color:#94a3b8;">${new Date().toLocaleTimeString()}</span>
    </div>
    <div style="margin-top:4px;"><b>${alert.symbol}</b>: ${alert.msg}</div>
    <div style="margin-top:4px; color:#fde047; font-size:11px;">${alert.control}</div>
  `;
  feed.prepend(card);
}

function triggerStrobe() {
  const strobe = document.getElementById('strobe');
  strobe.style.display = 'block';
  setTimeout(() => { strobe.style.display = 'none'; }, 4000);
}

function setDeskMode(mode) {
  deskMode = mode;
  setModeUI(mode);
  if (ws && ws.readyState === WebSocket.OPEN) {
    ws.send(JSON.stringify({ action: "TOGGLE_MODE", mode: mode }));
  }
}

function setModeUI(mode) {
  document.getElementById('btn-mode-sim').className = `mode-btn ${mode === 'SIMULATION' ? 'active-sim' : ''}`;
  document.getElementById('btn-mode-live').className = `mode-btn ${mode === 'LIVE' ? 'active-live' : ''}`;
  const pill = document.getElementById('market-status-pill');
  if (mode === 'LIVE') {
    pill.innerText = "MARKET CLOSED (SUNDAY - FROZEN AT FRIDAY CLOSE)";
    pill.className = "market-pill mp-closed";
  } else {
    pill.innerText = "SIMULATOR ACTIVE (ORDER-FLOW REPLAY)";
    pill.className = "market-pill mp-open";
  }
}

function resetKillSwitch() {
  if (ws) ws.send(JSON.stringify({ action: "RESET_KILL_SWITCH", symbol: currentSymbol }));
}

function adjustClipCap(val) {
  if (ws) ws.send(JSON.stringify({ action: "UPDATE_CLIP_CAP", symbol: currentSymbol, value: val }));
}

function injectAnomaly(kind) {
  if (ws) ws.send(JSON.stringify({ action: "SIMULATE_ANOMALY", symbol: currentSymbol, kind: kind }));
}

function exportAuditTrail() {
  if (auditHistory.length === 0) {
    alert("No risk alerts logged in this session yet.");
    return;
  }
  let csv = "data:text/csv;charset=utf-8,Time,Symbol,Severity,Typology,Details,Mitigation\\n";
  auditHistory.forEach(a => {
    csv += `"${a.time}","${a.symbol}","${a.severity}","${a.typology}","${a.msg}","${a.control}"\\n`;
  });
  const link = document.createElement("a");
  link.setAttribute("href", encodeURI(csv));
  link.setAttribute("download", `surveillance_audit_${Date.now()}.csv`);
  document.body.appendChild(link);
  link.click();
  document.body.removeChild(link);
}

window.onload = connectWS;
</script>
</body>
</html>
""")

Overwriting surveillance_desk.py


In [ ]:
import socket
import uvicorn
import nest_asyncio
nest_asyncio.apply()

def get_local_ip():
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    try:
        s.connect(('10.255.255.255', 1))
        ip = s.getsockname()[0]
    except Exception:
        ip = '127.0.0.1'
    finally:
        s.close()
    return ip

def find_available_port(start=8050):
    for port in range(start, start + 50):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('127.0.0.1', port)) != 0:
                return port
    return start

local_ip = get_local_ip()
port = find_available_port(8050)

print("=" * 65)
print(f"🖥️  DESKTOP (LOCALHOST): http://localhost:{port}")
print(f"📱 MOBILE / WI-FI NETWORK: http://{local_ip}:{port}")
print("=" * 65)

# Binding to 0.0.0.0 allows any phone/laptop on your Wi-Fi to load the UI
config = uvicorn.Config("surveillance_desk:app", host="0.0.0.0", port=port, log_level="info")
server = uvicorn.Server(config)
await server.serve()

🖥️  DESKTOP (LOCALHOST): http://localhost:8051
📱 MOBILE / WI-FI NETWORK: http://192.168.29.211:8051
Fetching latest closing prices from exchange...


INFO:     Started server process [34224]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8051 (Press CTRL+C to quit)
INFO:     127.0.0.1:60514 - "WebSocket /ws" [accepted]
INFO:     connection open


INFO:     192.168.29.197:55826 - "GET / HTTP/1.1" 200 OK


INFO:     192.168.29.197:55830 - "WebSocket /ws" [accepted]
INFO:     connection open


INFO:     192.168.29.197:55826 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     192.168.29.197:51234 - "GET / HTTP/1.1" 200 OK


...

INFO:     192.168.29.197:55158 - "WebSocket /ws" [accepted]
INFO:     connection open


INFO:     192.168.29.197:55138 - "GET / HTTP/1.1" 200 OK


INFO:     192.168.29.197:51640 - "WebSocket /ws" [accepted]
INFO:     connection open


INFO:     192.168.29.197:51628 - "GET / HTTP/1.1" 200 OK


INFO:     192.168.29.197:48706 - "WebSocket /ws" [accepted]
INFO:     connection open


In [1]:
import os

current_dir = os.getcwd()
print("=" * 60)
print(f"YOUR PROJECT FOLDER PATH IS:\n{current_dir}")
print("=" * 60)

YOUR PROJECT FOLDER PATH IS:
C:\Users\HP


In [5]:
import os

# 1. requirements.txt
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("""fastapi>=0.100.0
uvicorn[standard]>=0.23.0
websockets>=11.0
yfinance>=0.2.28
nest-asyncio>=1.5.6
""")

# 2. .gitignore
with open(".gitignore", "w", encoding="utf-8") as f:
    f.write("""__pycache__/
*.py[cod]
*$py.class
.ipynb_checkpoints/
.env
venv/
*.csv
""")

# 3. README.md
with open("README.md", "w", encoding="utf-8") as f:
    f.write("""# Cross-Asset Real-Time Market Surveillance & Electronic Trading Risk Desk

An interactive, production-grade electronic trading risk management and market surveillance console built with **FastAPI**, **WebSockets**, and **Chart.js**.

The desk monitors a 17-instrument global perimeter across US Equities, Indian NSE, Foreign Exchange, and Commodities, running sub-second statistical anomaly checks and enforcing automated in-flight risk safeguards.

---

## Key Features

- **Multi-Asset Telemetry**: Real-time sub-second price drift, rolling 1-minute volume, and statistical volatility calibration via trailing ATR-14.
- **Quantitative Abuse & Anomaly Detection**:
  - **Flash Price Dislocation**: Alerts and locks sessions when price drifts exceed 2.5x ATR-14.
  - **Volume Bursts**: Flags volume spikes at Z >= +4.0σ and clamps allowable clip-caps from 2,500 down to 500 units.
  - **Spoofing & Layering Signatures**: Detects high cancel-to-replace ratios (>= 90%) with ultra-short quote lifespans (< 200ms) and engages gateway throttling.
- **Level-2 (L2) Order Book Depth Ladder**: Live top-5 bid/ask ladder with visual depth bars and real-time Order Book Imbalance (OBI) metrics.
- **Raw FIX 4.2 Protocol Stream**: Live gateway logging of institutional tags:
  - `35=D` (New Order Single)
  - `35=F` (Order Cancel Request)
  - `35=8` (Execution Report)
- **Compliance Audit Logging**: Real-time dispatch ledger with one-click timestamped CSV export.
- **Responsive Web & Mobile Console**: Accessible from any desktop or mobile browser on the local network.

---

## Quick Start

```bash
pip install -r requirements.txt
uvicorn surveillance_desk:app --host 0.0.0.0 --port 8050

_IncompleteInputError: incomplete input (1985670007.py, line 25)

In [6]:
%%writefile requirements.txt
fastapi>=0.100.0
uvicorn[standard]>=0.23.0
websockets>=11.0
yfinance>=0.2.28
nest-asyncio>=1.5.6

Overwriting requirements.txt


In [7]:
%%writefile .gitignore
__pycache__/
*.py[cod]
*$py.class
.ipynb_checkpoints/
.env
venv/
*.csv

Writing .gitignore


In [8]:
%%writefile README.md
# Cross-Asset Real-Time Market Surveillance & Electronic Trading Risk Desk

An interactive, high-fidelity electronic trading risk management and market surveillance console built with **FastAPI**, **WebSockets**, and **Chart.js**.

The desk monitors a 17-instrument global perimeter across US Equities, Indian NSE, Foreign Exchange, and Commodities, running sub-second statistical anomaly checks and enforcing automated in-flight risk safeguards.

---

## Key Features

- **Multi-Asset Telemetry**: Real-time sub-second price drift, rolling 1-minute volume, and statistical volatility calibration via trailing ATR-14.
- **Quantitative Abuse & Anomaly Detection**:
  - **Flash Price Dislocation**: Alerts and locks sessions when price drifts exceed 2.5x ATR-14.
  - **Volume Bursts**: Flags volume spikes at Z >= +4.0σ and dynamically clamps allowable clip-caps from 2,500 down to 500 units.
  - **Spoofing & Layering Signatures**: Detects high cancel-to-replace ratios (>= 90%) with ultra-short quote lifespans (< 200ms) and engages gateway throttling.
- **Level-2 (L2) Order Book Depth Ladder**: Live top-5 bid/ask ladder with visual depth bars and real-time Order Book Imbalance (OBI) tracking.
- **Raw FIX 4.2 Protocol Stream**: Live gateway logging of institutional tags:
  - `35=D` (New Order Single)
  - `35=F` (Order Cancel Request)
  - `35=8` (Execution Report)
- **Compliance Audit Logging**: Real-time dispatch ledger with one-click timestamped CSV export.
- **Responsive Web & Mobile Console**: Accessible from any desktop or mobile browser on the local network.

---

## Tech Stack

- **Backend**: Python 3.10+, FastAPI, WebSockets, yfinance, asyncio
- **Frontend**: Vanilla JavaScript (ES6+), HTML5/CSS3, Chart.js, Web Speech API
- **Protocol**: Raw FIX 4.2 string serialization

---

## Quick Start

```bash
pip install -r requirements.txt
uvicorn surveillance_desk:app --host 0.0.0.0 --port 8050

Overwriting README.md


In [ ]:
git remote add origin https://github.com/karthikdas123/surveillance-desk.git
git branch -M main
git push -u origin main